# Results: saintes-2026-02

Reads only `processed/`, `outputs/` and the Filtered Cube written by the pipeline. Run the pipeline first:

```
uv run flood-monitoring run saintes-2026-02
```

In [ ]:
import json

import folium
import matplotlib.pyplot as plt
import numpy as np
import rasterio
import rioxarray
import xarray as xr
from folium.plugins import SideBySideLayers
from rasterio.warp import Resampling, calculate_default_transform, reproject

from flood_monitoring.composite import render_rgb
from flood_monitoring.config import load_event_config
from flood_monitoring.detection import to_db
from flood_monitoring.event import acquisition_dates
from flood_monitoring.paths import event_paths

EVENT = "saintes-2026-02"
config = load_event_config(EVENT)
paths = event_paths(EVENT)
summary = json.loads(paths.summary.read_text())
event_start, peak = summary["event_start"], summary["peak"]

with xr.open_dataset(paths.filtered_cube) as cube:
    dates = acquisition_dates(cube["time"].values)
    vv_db = to_db(cube["VV"].values, cube.attrs.get("nodata"))
print(f"{len(dates)} Acquisitions, Event Start {event_start}, Peak {peak}")

## 1. The Series

In [ ]:
fig, ax = plt.subplots(figsize=(10, 1.8))
when = np.array(dates, dtype="datetime64[D]")
ax.plot(when, np.zeros(len(when)), "|", markersize=18, color="0.3", label="Acquisition")
ax.axvline(np.datetime64(event_start), color="tab:red", label=f"Event Start {event_start}")
ax.axvline(np.datetime64(peak), color="tab:blue", label=f"Peak {peak}")
ax.set_yticks([])
ax.legend(loc="upper left", ncol=3, frameon=False, bbox_to_anchor=(0, 1.6))
fig.autofmt_xdate()
plt.show()
print(f"{dates[0]} to {dates[-1]}, {len(dates)} Acquisitions, every {(when[1] - when[0]).astype(int)} days")

## 2. The Drop

VV backscatter (dB) over the Series for a sample of pixels inside the Flood Extent, against a sample of land that never flooded (not in the Flood Extent, Permanent Water or any Water Mask). Flooded pixels fall sharply at the Event Start.

In [ ]:
def read_mask(path):
    with rioxarray.open_rasterio(path) as da:
        return da.values[0].astype(bool)

extent = read_mask(paths.flood_extent)
permanent = read_mask(paths.permanent_water)
ever_water = np.any([read_mask(p) for p in sorted(paths.water_masks.glob("*.tif"))], axis=0)
dry = ~(extent | permanent | ever_water)

rng = np.random.default_rng(0)
def sample(mask, n=2000):
    ys, xs = np.nonzero(mask)
    pick = rng.choice(len(ys), size=min(n, len(ys)), replace=False)
    return vv_db[:, ys[pick], xs[pick]]

flooded, land = sample(extent), sample(dry)
print(f"{flooded.shape[1]} flooded pixels, {land.shape[1]} never-flooded land pixels")

fig, ax = plt.subplots(figsize=(10, 4))
for series, label, color in ((flooded, "Flood Extent", "tab:blue"), (land, "never-flooded land", "tab:green")):
    ax.plot(when, np.nanmean(series, axis=1), color=color, label=f"{label}, mean")
    ax.plot(when, np.nanmedian(series, axis=1), color=color, ls="--", label=f"{label}, median")
ax.axvline(np.datetime64(event_start), color="tab:red", label="Event Start")
ax.set_ylabel("VV backscatter (dB)")
ax.legend(ncol=2, frameon=False)
fig.autofmt_xdate()
plt.show()

## 3. Water Masks and flooded area

In [ ]:
mask_files = sorted(paths.water_masks.glob("*.tif"))
fig, axes = plt.subplots(1, len(mask_files), figsize=(3.2 * len(mask_files), 3.4), squeeze=False)
for ax, f in zip(axes[0], mask_files):
    ax.imshow(read_mask(f), cmap="Blues", interpolation="nearest")
    ax.set_title(f.stem + (" (Peak)" if f.stem == peak else ""))
    ax.axis("off")
plt.show()

areas = summary["flooded_area_km2"]
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(list(areas), list(areas.values()), color=["tab:blue" if d == peak else "0.6" for d in areas])
ax.set_ylabel("flooded area (km²)")
fig.autofmt_xdate()
plt.show()
print(f"Flood Extent {summary['flood_extent_km2']} km², Permanent Water {summary['permanent_water_km2']} km²")

## 4. Flood Extent over the After Composite

The After Composite is the Peak, drawn as red = VV, green = VH, blue = VV minus VH (ranges from the Event config).

In [ ]:
RANGES = config.composite.as_dict()
NODATA = -9999.0

def wgs84(array, src, nodata):
    """Reproject (bands, y, x) to EPSG:4326; returns the array and [[south, west], [north, east]]."""
    t, w, h = calculate_default_transform(src.crs, "EPSG:4326", src.width, src.height, *src.bounds)
    out = np.full((array.shape[0], h, w), nodata, dtype=array.dtype)
    reproject(array, out, src_transform=src.transform, src_crs=src.crs, dst_transform=t,
              dst_crs="EPSG:4326", resampling=Resampling.nearest, dst_nodata=nodata)
    (west, north), (east, south) = t * (0, 0), t * (w, h)
    return out, [[south, west], [north, east]]

def composite_image(path):
    with rasterio.open(path) as src:
        comp, bounds = wgs84(src.read(), src, NODATA)
    alpha = np.where((comp == NODATA).all(axis=0), 0, 255).astype("uint8")
    return np.dstack([render_rgb(comp, RANGES), alpha]), bounds

def extent_image():
    with rasterio.open(paths.flood_extent) as src:
        # 255 as nodata: with 0 (also the "dry" value) rasterio marks most of the output as flooded
        data, bounds = wgs84(src.read(1)[None], src, 255)
    rgba = np.zeros((*data.shape[1:], 4), "uint8")
    rgba[data[0] == 1] =(0, 220, 255, 220)  # cyan: contrasts with the red/green/blue Composite
    return rgba, bounds

west, south, east, north = config.area_of_interest
center = [(south + north) / 2, (west + east) / 2]
before_img, bounds = composite_image(paths.before_composite)
after_img, _ = composite_image(paths.after_composite)
extent_img, extent_bounds = extent_image()

m = folium.Map(center, zoom_start=12, control_scale=True)
folium.raster_layers.ImageOverlay(after_img, bounds=bounds, name=f"After Composite ({peak})").add_to(m)
folium.raster_layers.ImageOverlay(extent_img, bounds=extent_bounds, name="Flood Extent", zindex=500).add_to(m)
folium.LayerControl().add_to(m)
m

## 5. Before / After

Drag the divider: Before Composite on the left, After Composite on the right. The Flood Extent sits in the After pane, so it only shows right of the divider; toggle it from the layer control.

Each overlay gets its own Leaflet pane, and `getContainer` is patched to return that pane: `SideBySideLayers` expects tile layers, which have a container, where image overlays do not.

In [ ]:
from branca.element import Element, MacroElement
from jinja2 import Template

PATCH = "L.ImageOverlay.prototype.getContainer = function(){ return this.getPane(); };"
PANES = """
{% macro script(this, kwargs) %}
  var m = {{ this._parent.get_name() }};
  ['before', 'after'].forEach(function(n){ m.createPane(n).style.zIndex = 300; });
{% endmacro %}
"""

split = folium.Map(center, zoom_start=12, tiles="OpenStreetMap", control_scale=True)
split.get_root().script.add_child(Element(PATCH))
panes = MacroElement()
panes._template = Template(PANES)
panes.add_to(split)

before = folium.raster_layers.ImageOverlay(
    before_img, bounds=bounds, name=f"Before Composite ({summary['before']})", pane="before").add_to(split)
after = folium.raster_layers.ImageOverlay(
    after_img, bounds=bounds, name=f"After Composite ({peak})", pane="after").add_to(split)
folium.raster_layers.ImageOverlay(
    extent_img, bounds=extent_bounds, name="Flood Extent", pane="after", zindex=500).add_to(split)
SideBySideLayers(before, after).add_to(split)
folium.LayerControl().add_to(split)
split